# Barrier Options

Barrier options are **path-dependent** derivatives whose payoff depends not
only on the terminal spot price but also on whether the underlying touches
(or fails to touch) a pre-specified **barrier level** $H$ during the
contract's life.  They are activated (**knock-in**) or extinguished
(**knock-out**) by this barrier event, which makes them cheaper than the
corresponding vanilla option and a popular tool for tailoring directional
views.

A barrier contract is fully specified by four orthogonal choices:

| Choice | Values |
|---|---|
| Option type | CALL / PUT |
| Direction   | UP (barrier above spot) / DOWN (barrier below spot) |
| Action      | IN (activates on touch) / OUT (extinguishes on touch) |
| Monitoring  | CONTINUOUS / DISCRETE |

That gives **eight** standard barrier types (`{call, put} × {up, down} ×
{in, out}`) and two monitoring conventions, plus an optional **rebate**
paid on knock-out (or as compensation for a knock-in that never activated).

**Roadmap**

1. Notebook setup
2. Anatomy of a barrier option (the eight types, monitoring, rebates)
3. Continuous monitoring — pricing across the four engines (European)
4. American exercise (continuous monitoring)
5. Greeks
6. Discrete monitoring — pricing across the four engines (European)
7. American exercise with discrete monitoring
8. Summary

## 1) Notebook Setup

We'll use the `derivatives_pricing` package throughout.  Key objects for
barrier pricing:

| Class / Enum | Purpose |
|---|---|
| `BarrierSpec` | Barrier contract terms (strike, barrier, direction, action, monitoring, rebate) |
| `BarrierDirection` | `UP` / `DOWN` |
| `BarrierAction` | `IN` / `OUT` |
| `BarrierMonitoring` | `CONTINUOUS` / `DISCRETE` |
| `RebateTiming` | `AT_HIT` / `AT_EXPIRY` |
| `OptionValuation` | Dispatcher — routes `(method, exercise)` to the right engine |
| `PricingMethod` | `BSM` (analytical), `BINOMIAL`, `PDE_FD`, `MONTE_CARLO` |
| `GBMProcess` / `SimulationConfig` | GBM path simulator (used by Monte Carlo) |

In [ ]:
from dataclasses import replace as dc_replace
import datetime as dt
import math
import warnings

import derivatives_pricing as dp

In [ ]:
# ── Market / contract parameters ──────────────────────────────────────
pricing_date = dt.datetime(2025, 1, 1)
maturity = dt.datetime(2026, 1, 1)

S0 = 100.0  # spot
K = 100.0  # strike (ATM)
sigma = 0.25  # volatility
r = 0.05  # risk-free rate
q = 0.02  # continuous dividend yield
T = 1.0  # year fraction (display only)

# Continuous DOWN barrier: safely below spot so the DO call is alive at
# inception (and a DI call needs a downward move to activate).
H_DOWN = 85.0

curve_r = dp.DiscountCurve.flat(r, end_time=2.0)
curve_q = dp.DiscountCurve.flat(q, end_time=2.0)

market_data = dp.MarketData(pricing_date, curve_r, currency="USD")
underlying = dp.UnderlyingData(
    initial_value=S0,
    volatility=sigma,
    market_data=market_data,
    dividend_curve=curve_q,
)

print(f"Spot {S0},  Strike {K},  σ {sigma},  r {r},  q {q},  T {T}")
print(f"DOWN barrier {H_DOWN}  (alive at inception: spot {S0} > {H_DOWN})")

Spot 100.0,  Strike 100.0,  σ 0.25,  r 0.05,  q 0.02,  T 1.0
DOWN barrier 85.0  (alive at inception: spot 100.0 > 85.0)


## 2) Anatomy of a Barrier Option

### 2.1 The eight types

The payoff at maturity is:

- **Knock-out (OUT)**: $\text{vanilla payoff}\;\text{if barrier never touched}$, else $\text{rebate}$
- **Knock-in (IN)**: $\text{vanilla payoff}\;\text{if barrier ever touched}$, else $\text{rebate (or 0)}$

Combined with `{call, put} × {up, down}`, this gives the standard nomenclature:

| Name | Direction | Action | Behaviour |
|---|---|---|---|
| Up-and-out (UO)   | UP   | OUT | Killed if spot rises to $H$ |
| Up-and-in (UI)    | UP   | IN  | Activated if spot rises to $H$ |
| Down-and-out (DO) | DOWN | OUT | Killed if spot falls to $H$ |
| Down-and-in (DI)  | DOWN | IN  | Activated if spot falls to $H$ |

Each variant exists in CALL and PUT flavours.

### 2.2 In-out parity

For a **European** barrier without rebate, a fundamental identity holds:

$$V_{\text{vanilla}} = V_{\text{KI}} + V_{\text{KO}}$$

Either the barrier is hit (the KI activates and pays vanilla, the KO is
dead) or it is not (the KO pays vanilla, the KI pays nothing) — never
both, never neither.  The two events partition the sample space, so their
expected discounted payoffs sum to the vanilla.

**With rebates** (KO at-expiry rebate $R$ paid if hit; KI at-expiry rebate
$R$ paid if *not* hit) the parity generalises to:

$$V_{\text{vanilla}} + R \cdot \text{df}(0,T) = V_{\text{KI}} + V_{\text{KO}}$$

The reason for the extra $R \cdot \text{df}(0, T)$ is that one of the 
barrier contracts will pay the rebate at maturity.

This identity holds only:
- For **European exercise** (American breaks because the optimal
  stopping rule depends on whether the option is knocked in).
- With both barrier specs sharing strike, barrier, direction, and
  monitoring.
- For at-expiry rebates on both legs.

We will use this identity later when pricing European KI on the PDE
engine.

### 2.3 Monitoring

- **Continuous**: the barrier is checked at every instant.  Closed-form
  formulas exist under GBM (Reiner-Rubinstein 1991).
- **Discrete**: the barrier is checked only at scheduled observation dates
  (e.g. daily or monthly).  Discrete monitoring strictly **reduces** hit
  probability vs continuous monitoring with the same nominal barrier — there
  are simply fewer chances to be killed (knock-out) or activated (knock-in).
  Discrete KO calls are therefore **more** valuable than their continuous
  equivalent; discrete KI calls are **less** valuable.

### 2.4 Rebates

Optionally a barrier contract may pay a fixed cash **rebate** $R$:

- **Knock-out + rebate**: $R$ paid when the barrier is touched, either
  immediately (`AT_HIT`) or at expiry (`AT_EXPIRY`).  Compensation for the
  contract dying.
- **Knock-in + rebate**: $R$ paid at expiry **only if the barrier is never
  touched** (i.e. the option never activated).  Compensation for the
  contract never coming into being.

## 3) Continuous European Pricing — Four Engines

We start with the simplest case: **European exercise**, **continuous
monitoring**, no rebate.  All four engines apply.  We'll use a
**down-and-out call** ($H = 85$, alive at inception) as the running
example and compare against its **down-and-in** twin.

In [ ]:
# ── European down-out and down-in call specs ──────────────────────────
spec_doc = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_DOWN,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_dic = dc_replace(spec_doc, action=dp.BarrierAction.IN)

# Vanilla European call for the in-out parity check.
spec_vanilla_call = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
)

### 3.1 BSM analytical (Reiner-Rubinstein)

Under GBM dynamics, every European single-barrier option without
discontinuous dividends admits a **closed-form** price.  The 
`derivatives-pricing` package implements the full Reiner-Rubinstein
(1991) formulas for all eight standard types. The price decomposes into
four building-block terms (labelled $A, B, C, D$) of two shapes.

**Vanilla shape** (terms $A, B$) — standard Black-Scholes form:

$$\phi\,S\,e^{-qT}\,\Phi(\phi z) - \phi\,K\,e^{-rT}\,\Phi(\phi z - \phi\sigma\sqrt{T})$$

**Reflected shape** (terms $C, D$) — image of the vanilla shape about $H$,
carrying an $(H/S)^{2\lambda}$ factor that arises from the running-min /
running-max distribution of GBM (method of images about the barrier):

$$\phi\,S\,e^{-qT}\,(H/S)^{2\lambda}\,\Phi(\eta z) - \phi\,K\,e^{-rT}\,(H/S)^{2\lambda - 2}\,\Phi(\eta z - \eta\sigma\sqrt{T})$$

with $\lambda = (r - q + \sigma^2/2)/\sigma^2$ and $\phi, \eta \in \{+1, -1\}$
chosen by `(option_type, direction)`.  Within each pair, one term is
**vanilla-like** (log-arg uses the strike, e.g. $\log(S/K)$ or $\log(H^2/(SK))$)
and the other is **truncated at the barrier** (log-arg uses $H$ alone, e.g.
$\log(S/H)$ or $\log(H/S)$):

| Term | Shape | Log-arg flavour |
|---|---|---|
| $A$ | vanilla   | vanilla-like ($\log(S/K)$) |
| $B$ | vanilla   | truncated at barrier ($\log(S/H)$) |
| $C$ | reflected | vanilla-like, reflected ($\log(H^2/(SK))$) |
| $D$ | reflected | truncated at barrier, reflected ($\log(H/S)$) |

We won't reproduce the per-type term selection here — see
Reiner-Rubinstein (1991) "Breaking Down the Barriers", *Risk* **4**(8),
28–35, or Hull §26.9.

The package selects the appropriate combination of the four terms based on
`(direction, action, option_type)` and the spot-vs-barrier-vs-strike
ordering.  Pricing is one O(1) call to standard normal CDFs — this is the
fastest engine by orders of magnitude.

In [ ]:
# ── BSM analytical ────────────────────────────────────────────────────
bsm_doc = dp.OptionValuation(underlying, spec_doc, dp.PricingMethod.BSM).present_value()
bsm_dic = dp.OptionValuation(underlying, spec_dic, dp.PricingMethod.BSM).present_value()
bsm_vanilla = dp.OptionValuation(
    underlying, spec_vanilla_call, dp.PricingMethod.BSM
).present_value()

print("── BSM Analytical (continuous European, DOWN call) ──")
print(f"  Down-and-out:    {bsm_doc:.6f}")
print(f"  Down-and-in:     {bsm_dic:.6f}")
print(f"  Vanilla (sum):   {bsm_doc + bsm_dic:.6f}")
print(f"  Vanilla (true):  {bsm_vanilla:.6f}")
print(f"  In-out parity error: {abs(bsm_doc + bsm_dic - bsm_vanilla):.6f}")

── BSM Analytical (continuous European, DOWN call) ──
  Down-and-out:    9.923414
  Down-and-in:     1.200348
  Vanilla (sum):   11.123762
  Vanilla (true):  11.123762
  In-out parity error: 0.000000


The in-out parity holds to machine precision, confirming the analytical
formulas are internally consistent.

### 3.2 Where do the analytical rebate formulas come from?

Rebates are valued separately from the option payoff and added in:

- **KO at-hit rebate** ($R$ paid the instant the barrier is first touched):
  this is a **one-touch digital** — its present value is the expected
  discounted value of $R$ paid at the first hit time $\tau_B$, conditional on
  $\tau_B \le T$:
  $$\text{PV}_{\text{at-hit}} = R \cdot E\!\left[e^{-r\tau_B}\,\mathbb{1}_{\tau_B \le T}\right]$$
  The Reiner-Rubinstein closed form evaluates this in terms of two normal CDFs of the
  running-min/running-max distribution of GBM.  Practically: it's a known
  formula derived from the joint distribution of $(\tau_B, S_T)$.

- **KO at-expiry rebate** ($R$ paid at maturity if the barrier was ever hit):
  this is the **first-passage probability** times a discount factor:
  $$\text{PV}_{\text{at-expiry}}^{\text{KO}} = R \cdot e^{-rT}\, P(\tau_B \le T)$$
  where $P(\tau_B \le T)$ is the **reflection-principle**
  probability for Brownian motion with drift $\mu = r - q - \tfrac{1}{2}\sigma^2$:
  $$P(\tau_B \le T) = \Phi(\eta a) + (H/S)^{2\mu/\sigma^2}\, \Phi(\eta b)$$
  with $a = (\ln(H/S) - \mu T)/(\sigma\sqrt T)$, $b = (\ln(H/S) + \mu T)/(\sigma\sqrt T)$
  and $\eta = +1$ for DOWN, $-1$ for UP.

- **KI at-expiry rebate** ($R$ paid at maturity only if the barrier was
  *never* hit — i.e. the option never activated): this is the **no-touch
  probability** times a discount factor:
  $$\text{PV}_{\text{at-expiry}}^{\text{KI}} = R \cdot e^{-rT}\,\bigl[1 - P(\tau_B \le T)\bigr]$$

All three are exact under GBM and follow from the same first-passage theory
of Brownian motion with drift (the running-min/running-max joint distribution
obtained via the reflection principle).  Implementation lives in
`barrier_analytical._rebate_*`.

### 3.3 Binomial — Boyle-Lau step adjustment

A naive CRR binomial tree sized at $N$ steps places spot nodes at
$S_0 \cdot u^{j} d^{N-j}$.  The barrier $H$ generally falls **between** two
adjacent spot levels; the effective barrier used is shifted **outward** 
from spot relative to the true $H$, enlarging the alive
zone — KO is systematically **over-priced** (and KI under-priced) until 
a tree node lands exactly on $H$.  This is a $O(1/\sqrt{N})$ bias
dominated by where $H$ falls in the tree geometry rather than by tree
resolution: doubling $N$ doesn't reliably halve the error, it usually just
shifts the misalignment to a different sub-pixel.

**Boyle-Lau (1994)** fixes this: pick $N$ such that one tree layer of nodes
lies **exactly on the barrier**.  The condition is:

$$N_i = \left\lfloor \frac{i^2 \sigma^2 T}{(\ln(H/S))^2} \right\rfloor \quad \text{for } i = 1, 2, 3, \dots$$

Each $N_i$ produces a different CRR tree; for the smallest $N_i$ that is
$\ge$ the user's base `num_steps`, the tree has a layer landing essentially
on $H$.  Once aligned, the discretisation bias collapses to standard
$O(1/N)$ behaviour and convergence is much smoother.

Picking the right $N$ happens automatically inside
`_BinomialBarrierValuation._resolve_effective_num_steps()`.  If the
alignment requires more than `max(1000, 5 × num_steps)` tree time steps
(typical for barriers extremely close to spot), the package warns and
**caps at that maximum** rather than letting memory grow unboundedly.
In the capped regime the tree runs *without* barrier alignment and only
achieves the slow $O(1/\sqrt{N})$ Boyle-Lau bias — at which point you
should switch engines (PDE_FD is recommended).

Regarding how the binomial engine prices KOs/KIs, once the effective number
of steps is resolved:

**Knock-out approach.** Standard backward induction, but at every monitored
step (every step under continuous monitoring) any node at or past the
barrier is **killed** — set to the rebate/discounted rebate value (or 0).
Roll back from maturity to today.

**Knock-in approach.** A two-state recursion: track $V$ on each node in two
states — *active* (already knocked in, behaves as vanilla) and *inactive*
(not yet hit, value is the still-live KI).  At hit-eligible nodes, the
inactive value transitions to the active value.  This handles American KI
and KI rebates without relying on parity (which would fail for American
exercise).

In [ ]:
# ── Binomial ──────────────────────────────────────────────────────────
binom_doc = dp.OptionValuation(underlying, spec_doc, dp.PricingMethod.BINOMIAL).present_value()
binom_dic = dp.OptionValuation(underlying, spec_dic, dp.PricingMethod.BINOMIAL).present_value()

print("── Binomial (continuous European, DOWN call) ──")
print(
    f"  Down-and-out:  {binom_doc:.6f}   (BSM: {bsm_doc:.6f},  diff {binom_doc - bsm_doc:+.6f}  "
    f"({(binom_doc - bsm_doc) / bsm_doc:+.2%}))"
)
print(
    f"  Down-and-in:   {binom_dic:.6f}   (BSM: {bsm_dic:.6f},  diff {binom_dic - bsm_dic:+.6f}  "
    f"({(binom_dic - bsm_dic) / bsm_dic:+.2%}))"
)

── Binomial (continuous European, DOWN call) ──
  Down-and-out:  9.926217   (BSM: 9.923414,  diff +0.002804  (+0.03%))
  Down-and-in:   1.199774   (BSM: 1.200348,  diff -0.000574  (-0.05%))


### 3.4 PDE_FD — truncated grid for KO, in-out parity for KI

The Black-Scholes PDE

$$\frac{\partial V}{\partial t} + (r - q)\,S\frac{\partial V}{\partial S}
 + \tfrac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} - r V = 0$$

is solved on a finite spot grid by stepping backward in time from the
terminal payoff to today.  For a vanilla option we choose $[S_{\min},
S_{\max}]$ wide enough that the boundary conditions there are negligible.

**Continuous knock-out: truncate the grid at the barrier.**  Instead of
using the full vanilla domain, we use:

- DOWN-OUT: $[H, S_{\max}]$ with the **Dirichlet** boundary $V(H, t)$
  at the lower edge equal to:
  - $0$ if no rebate;
  - $R$ if the rebate is paid AT_HIT (cash on knock-out, no further
    discounting);
  - $R \cdot \text{df}(t, T)$ if paid AT_EXPIRY (once dead the contract
    just pays $R$ at $T$, so its value at the kill time $t$ is the
    discounted rebate).
- UP-OUT:   $[S_{\min}, H]$ with the same boundary value at the upper edge.

Truncation eliminates the discontinuity in $V$ at $H$ (where the option
is killed) from the *interior* of the grid — it now lives on the boundary
where the discretisation handles it cleanly.  This gives the PDE engine
good accuracy and stability for KO barriers, even for barriers very
close to spot.

**Continuous knock-in (European):** We price European KIs using in-out parity. 
Without rebates, we use:

$$V_{\text{KI}} = V_{\text{vanilla}} - V_{\text{KO}}$$

Both right-hand-side prices are themselves PDE solves: the vanilla on a
wide grid, the KO on the truncated grid.  When a KI **rebate** is included, parity
generalises (as in §2.2) to 

$$V_{\text{KI}} = V_{\text{vanilla}} + R \cdot \text{df}_T - V_{\text{KO}}$$

Note the above only holds for European options, for Americans we directly solve a 
two-surface coupled PDE.

**Default scheme.** Crank-Nicolson is used for continuous barriers
(smooth time march once we get past the initial maturity discontinuity).
`PDEParams.for_barriers(monitoring=BarrierMonitoring.CONTINUOUS)` ships with continuous defaults
of `1200 × 800` (spot × time) on a log-spot grid.

In [ ]:
# ── PDE_FD ────────────────────────────────────────────────────────────
pde_doc = dp.OptionValuation(underlying, spec_doc, dp.PricingMethod.PDE_FD).present_value()
pde_dic = dp.OptionValuation(underlying, spec_dic, dp.PricingMethod.PDE_FD).present_value()

print("── PDE_FD (continuous European, DOWN call) ──")
print(
    f"  Down-and-out:  {pde_doc:.6f}   (BSM: {bsm_doc:.6f},  diff {pde_doc - bsm_doc:+.6f}  "
    f"({(pde_doc - bsm_doc) / bsm_doc:+.2%}))"
)
print(
    f"  Down-and-in:   {pde_dic:.6f}   (BSM: {bsm_dic:.6f},  diff {pde_dic - bsm_dic:+.6f}  "
    f"({(pde_dic - bsm_dic) / bsm_dic:+.2%}))"
)

── PDE_FD (continuous European, DOWN call) ──
  Down-and-out:  9.923395   (BSM: 9.923414,  diff -0.000019  (-0.00%))
  Down-and-in:   1.200267   (BSM: 1.200348,  diff -0.000081  (-0.01%))


### 3.5 Monte Carlo — Brownian-bridge continuity correction

Naive MC for continuous barriers simulates GBM paths at discrete grid points
and marks a path as "hit" if any grid point breaches the barrier.  Because
spot can cross the barrier *between* grid points and recover before the next
observation, the naive check **underestimates the true hit probability**.
This propagates into the prices in opposite directions:

- **KO PV is biased high**: missed intra-step hits leave paths classified as
  alive when they should have been killed → too many paths contribute a
  vanilla payoff.
- **KI PV is biased low**: the same missed hits leave paths classified as
  not-yet-activated when they should have been knocked in → too few paths
  contribute a vanilla payoff.

The bias on the hit probability is $O(\sqrt{\Delta t})$, which dominates
standard MC sampling error unless $\Delta t$ is impractically small.

**Brownian-bridge correction.**  Conditional on the two endpoint spots
$S_i, S_{i+1}$ on a step of length $\Delta t$, the maximum (or minimum) of
log-spot follows a known reflected-Brownian distribution.  The probability
of the barrier being touched within the step (given both endpoints are on
the same side of $H$) is:

$$p_{\text{hit}}^{\text{step}} = \exp\!\left(-\frac{2\,\ln(H/S_i)\,\ln(H/S_{i+1})}{\sigma^2 \Delta t}\right)$$

(and exactly 1 if either endpoint already crosses).  This is the standard Brownian-bridge
crossing formula for log-GBM.  In `monte_carlo.py` we 
multiply through step-by-step to maintain a per-path **survival weight**:

$$\text{surv}^{(p)} = \prod_{i=0}^{N-1} \bigl(1 - p_{\text{hit},i}^{\text{step},(p)}\bigr)$$

where $(p)$ indexes the simulated path (one survival weight per path) and
$N$ is the number of time-grid steps.  The KO weight is `surv`; the KI
weight is `1 − surv`.  Each path's discounted vanilla payoff is multiplied
by its weight before averaging.  For European continuous barriers under GBM,
Brownian-bridge survival weighting removes the monitoring bias in the barrier
event itself, so ultra-fine monitoring grids are unnecessary.  The remaining
error is mainly MC sampling (and, for AT_HIT rebates, the midpoint
approximation to the unknown intra-step hit time).

Rebates are accumulated path-by-path along the same survival recursion
(AT_HIT discount uses the step midpoint as a fast-and-clean proxy for
the unknown intra-step hit time; AT_EXPIRY discounts to maturity).

In [ ]:
# ── Monte Carlo ───────────────────────────────────────────────────────
MC_PATHS = 200_000

sim_config = dp.SimulationConfig(
    paths=MC_PATHS,
    end_date=maturity,
    num_steps=52,  # weekly grid; bridge correction handles intra-step
)

gbm = dp.GBMProcess(
    market_data=market_data,
    process_params=dp.GBMParams(initial_value=S0, volatility=sigma, dividend_curve=curve_q),
    sim_config=sim_config,
)


mc_doc = dp.OptionValuation(gbm, spec_doc, dp.PricingMethod.MONTE_CARLO).present_value()
mc_dic = dp.OptionValuation(gbm, spec_dic, dp.PricingMethod.MONTE_CARLO).present_value()

print("── Monte Carlo (continuous European, DOWN call) ──")
print(
    f"  Down-and-out:  {mc_doc:.6f}   (BSM: {bsm_doc:.6f},  diff {mc_doc - bsm_doc:+.6f}  "
    f"({(mc_doc - bsm_doc) / bsm_doc:+.2%}))"
)
print(
    f"  Down-and-in:   {mc_dic:.6f}   (BSM: {bsm_dic:.6f},  diff {mc_dic - bsm_dic:+.6f}  "
    f"({(mc_dic - bsm_dic) / bsm_dic:+.2%}))"
)

── Monte Carlo (continuous European, DOWN call) ──
  Down-and-out:  9.965768   (BSM: 9.923414,  diff +0.042354  (+0.43%))
  Down-and-in:   1.200681   (BSM: 1.200348,  diff +0.000332  (+0.03%))


### 3.6 European comparison

All four engines should agree to within their respective discretisation
tolerances.  The analytical BSM is the ground truth.

In [ ]:
# ── European DO call: cross-engine table ───────────────────────────────
print(f"{'Engine':<14} {'DO call':>12} {'DI call':>12} {'Sum (vanilla)':>15}")
print("-" * 56)
print(f"{'BSM analytical':<14} {bsm_doc:>12.6f} {bsm_dic:>12.6f} {bsm_doc + bsm_dic:>15.6f}")
print(f"{'Binomial':<14} {binom_doc:>12.6f} {binom_dic:>12.6f} {binom_doc + binom_dic:>15.6f}")
print(f"{'PDE_FD':<14} {pde_doc:>12.6f} {pde_dic:>12.6f} {pde_doc + pde_dic:>15.6f}")
print(f"{'Monte Carlo':<14} {mc_doc:>12.6f} {mc_dic:>12.6f} {mc_doc + mc_dic:>15.6f}")
print(f"\nVanilla call (BSM): {bsm_vanilla:.6f}")

Engine              DO call      DI call   Sum (vanilla)
--------------------------------------------------------
BSM analytical     9.923414     1.200348       11.123762
Binomial           9.926217     1.199774       11.125992
PDE_FD             9.923395     1.200267       11.123662
Monte Carlo        9.965768     1.200681       11.166449

Vanilla call (BSM): 11.123762


## 4) American Exercise (Continuous Monitoring)

American barriers behave like American vanillas with the additional rule
"and not yet knocked-out / and already knocked-in".  Three of the four
engines support American exercise; **BSM analytical does not** because no
closed form exists.

We price a **down-and-in put** (DI put) in the example below (we choose 
a put because American early-exercise premium for a put can be meaningful).

In [ ]:
# ── American DI put ──────────────────────────────────────────────────
spec_dip_eu = dp.BarrierSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_DOWN,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.IN,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
spec_dip_am = dc_replace(spec_dip_eu, exercise_type=dp.ExerciseType.AMERICAN)

### 4.1 Binomial American

Identical Boyle-Lau step alignment as the European case (§3.3) — the tree
geometry doesn't depend on exercise style.  The two-state KI recursion
(§3.3) is augmented with an early-exercise comparison on the **active**
state at every step:

$$V_{\text{act}}(S, t) = \max\!\bigl(\text{intrinsic}(S),\; \text{discounted continuation}_{\text{act}}\bigr)$$

The **inactive** state has no exercise (the option hasn't activated yet)
so it just discounts continuation.  At hit-eligible nodes the inactive
value transitions into the active value.

In [ ]:
binom_dip_eu = dp.OptionValuation(
    underlying, spec_dip_eu, dp.PricingMethod.BINOMIAL
).present_value()
binom_dip_am = dp.OptionValuation(
    underlying, spec_dip_am, dp.PricingMethod.BINOMIAL
).present_value()

print("── Binomial (continuous DOWN-and-in put) ──")
print(f"  European:  {binom_dip_eu:.6f}")
print(f"  American:  {binom_dip_am:.6f}")
print(f"  Early-exercise premium:  {binom_dip_am - binom_dip_eu:+.6f}")

── Binomial (continuous DOWN-and-in put) ──
  European:  7.813432
  American:  8.147734
  Early-exercise premium:  +0.334303


### 4.2 PDE_FD American — two-surface coupled PDE for KI

**KO with American exercise.**  Solve the BS PDE on the truncated grid
(same domain as the European KO) but at every time step apply the
free-boundary condition $V \ge \text{intrinsic}$.  The package uses
**PSOR** (projected successive over-relaxation, `PDEEarlyExercise.GAUSS_SEIDEL`)
by default; a faster but less accurate `INTRINSIC` projection (single
`max(V, intrinsic)` sweep per time step, no inner iteration) is also
available — see notebook 06b §2 for the speed/accuracy trade-off.

**KI with American exercise — why parity fails.**  The European in-out
parity $V_{\text{KI}} = V_{\text{vanilla}} - V_{\text{KO}}$ relied on
both sides being held to maturity.  For American exercise, the optimal
stopping rule for the *vanilla* American (always alive) is different
from the optimal rule for the *KI American* (only exercisable once
knocked in).  So $V_{\text{KI}}^{\text{Am}} \neq V_{\text{vanilla}}^{\text{Am}} - V_{\text{KO}}^{\text{Am}}$.

**The two-surface coupled PDE.**  We solve two coupled grids
simultaneously:

- **$V_{\text{inact}}$** — the *inactive* surface: value while the
  barrier is *not yet* hit (no exercise allowed in this state since the
  option hasn't activated).
- **$V_{\text{act}}$** — the *active* surface: value once the barrier
  *has* been hit (a vanilla American option from that point onward).

At each time step we solve $V_{\text{act}}$ first as a vanilla-American
PDE on the full grid; then $V_{\text{inact}}$ on the inactive sub-grid
(interior of $[S_{\min}, H]$ for UP-IN, $[H, S_{\max}]$ for DOWN-IN)
with the boundary condition $V_{\text{inact}}(H, t) = V_{\text{act}}(H, t)$
(touching the barrier transitions you into the active state).  At
$t = 0$, the price is read off $V_{\text{inact}}$ at spot.

In [ ]:
pde_dip_eu = dp.OptionValuation(underlying, spec_dip_eu, dp.PricingMethod.PDE_FD).present_value()
pde_dip_am = dp.OptionValuation(underlying, spec_dip_am, dp.PricingMethod.PDE_FD).present_value()

print("── PDE_FD (continuous DOWN-and-in put) ──")
print(f"  European:  {pde_dip_eu:.6f}")
print(f"  American:  {pde_dip_am:.6f}")
print(f"  Early-exercise premium:  {pde_dip_am - pde_dip_eu:+.6f}")

── PDE_FD (continuous DOWN-and-in put) ──
  European:  7.810096
  American:  8.144658
  Early-exercise premium:  +0.334563


### 4.3 Monte Carlo American — Longstaff-Schwartz

Standard **Longstaff-Schwartz** (LSM) regresses continuation values onto
a polynomial basis in spot, then exercises wherever intrinsic exceeds
the regression's continuation estimate.  For barriers we adapt the per-
step backward-induction logic to the spec's `action`:

- **Knock-out** (`_knock_out_step_values` in `monte_carlo.py`): at each
  exercise date $t$ we form the alive+ITM mask (`alive = ~ever_hit`,
  `itm = (intrinsic > 0) & alive`) and **regress on those paths only**
  to estimate the continuation value.  Alive paths exercise where
  intrinsic > continuation; otherwise they discount the next-step value.
  Dead paths get either 0 (no rebate) or the appropriate rebate cashflow
  (immediate $R$ on the just-hit step for AT_HIT, or the discounted
  forward value $R \cdot \text{df}(t, T)$ for AT_EXPIRY).
- **Knock-in** (`_knock_in_step_values`): symmetric but inverted —
  regression runs on **active+ITM** paths (already knocked in and
  in-the-money).  Inactive paths just carry the discounted next-step
  value pathwise (no separate regression needed because there is no
  exercise decision until the option activates).

**Barrier-aware basis (KO only).**  For knock-outs, the basis can be
*enriched* with barrier-distance features so the fitted continuation
surface respects the barrier discontinuity at $H$.  This is controlled
by `MonteCarloParams.barrier_aware_basis` (default `True`) and is gated
inside `_knock_out_step_values` (`monte_carlo.py:1696`) — it is **not**
applied to knock-ins (there isn't a sharp continuation cliff as there is
for KOs).

**Caveat — DO puts and UO calls.**  Even with the barrier-aware basis,
LSM remains heavily downward-biased for **truncated-payoff** KO
Americans: a DO put or UO call kills the option in exactly the region
where its vanilla value is largest, so the rare paths that survive *and*
contribute meaningful intrinsic dominate the price but are sparsely
represented in any finite path sample.  The barrier-aware basis improves
the fit somewhat but doesn't close the bias.  Because of this, the
package emits a **warning at `__init__`** for American MC + DOP / UOC
specs (`monte_carlo.py:1649`), recommending PDE_FD instead.  Other
American barrier specs (DI put, UI call, DOC, UI put, ...) don't trigger
the warning — they converge to within typical LSM tolerances.

Our running example here is a DI put, which converges cleanly.

In [ ]:
mc_dip_eu = dp.OptionValuation(gbm, spec_dip_eu, dp.PricingMethod.MONTE_CARLO).present_value()
mc_dip_am = dp.OptionValuation(gbm, spec_dip_am, dp.PricingMethod.MONTE_CARLO).present_value()

print("── Monte Carlo (continuous DOWN-and-in put) ──")
print(f"  European:  {mc_dip_eu:.6f}")
print(f"  American:  {mc_dip_am:.6f}   (LSM)")
print(f"  Early-exercise premium:  {mc_dip_am - mc_dip_eu:+.6f}")

── Monte Carlo (continuous DOWN-and-in put) ──
  European:  7.824222
  American:  8.134961   (LSM)
  Early-exercise premium:  +0.310739


### 4.4 American comparison

In [ ]:
# Analytical European reference (only EU is closed-form)
bsm_dip_eu = dp.OptionValuation(underlying, spec_dip_eu, dp.PricingMethod.BSM).present_value()

print(f"{'Engine':<16} {'European':>12} {'American':>12} {'EE prem':>12}")
print("-" * 54)
print(f"{'BSM analytical':<16} {bsm_dip_eu:>12.6f} {'n/a':>12} {'n/a':>12}")
print(
    f"{'Binomial':<16} {binom_dip_eu:>12.6f} {binom_dip_am:>12.6f} "
    f"{binom_dip_am - binom_dip_eu:>+12.6f}"
)
print(f"{'PDE_FD':<16} {pde_dip_eu:>12.6f} {pde_dip_am:>12.6f} {pde_dip_am - pde_dip_eu:>+12.6f}")
print(f"{'Monte Carlo':<16} {mc_dip_eu:>12.6f} {mc_dip_am:>12.6f} {mc_dip_am - mc_dip_eu:>+12.6f}")

Engine               European     American      EE prem
------------------------------------------------------
BSM analytical       7.810181          n/a          n/a
Binomial             7.813432     8.147734    +0.334303
PDE_FD               7.810096     8.144658    +0.334563
Monte Carlo          7.824222     8.134961    +0.310739


## 5) Greeks

All five Greeks (delta, gamma, vega, theta, rho) are available across the
engine set as a whole, but not every greek is available on every engine.
The greek calculation method is selected automatically based on the
engine's capabilities, or can be overriden via `greek_calc_method`.
Monitoring affects availability at the margin: for example, numerical
theta is blocked for discretely monitored barriers because bumping the
pricing date changes the contractual monitoring schedule.

| Engine | Auto-selected greek method | Notes |
|---|---|---|
| BSM | Mostly NUMERICAL | delta/gamma/vega/rho use bump-and-reprice (NUMERICAL) by default; theta has a special auto-path that uses the BS PDE identity with the NUMERICAL spatial derivatives, rather than a time bump (former was empirically more accurate) |
| Binomial | TREE ($\delta/\gamma/\theta$)<br>NUMERICAL ($\rho$)| Reads delta/gamma off the lattice nodes (Hull §21); theta from the time slice |
| PDE_FD | GRID ($\delta/\gamma/\theta$)<br>NUMERICAL (vega/$\rho$) | Reads delta/gamma directly from the spatial grid; theta via the BS PDE identity $\theta = rV - (r-q)S\delta - \tfrac{1}{2}\sigma^2 S^2 \Gamma$ using those grid-derived $\delta/\Gamma$ |
| Monte Carlo | NUMERICAL | Bump and reprice using **common random numbers** (CRN) — same seed across bumps to control variance |

**Caveats specific to barriers**:

- The numerical spot bump for delta/gamma is **automatically capped** so
  the bumped spot stays on the same side of the barrier as $S_0$ — bumping
  across the barrier would price a different contract (alive
  vs dead).
- Binomial NUMERICAL greeks are blocked on barrier specs (except $\rho$) : 
  for continuous monitoring, bumping spot/vol/time
  re-runs Boyle-Lau and picks a different effective `num_steps` for each
  bump, so the central difference compares two different tree topologies.
- MC barrier gamma is blocked: the central-difference signal-to-noise
  ratio is too low for practical path counts.
- Discrete-monitoring barrier theta via NUMERICAL is blocked across
  engines: bumping the pricing date forces re-resolution of the
  monitoring schedule (a different contract on the bumped side).
  Use TREE/GRID instead.
- The default time bump for barrier NUMERICAL (bump and revalue) theta is **7 calendar days** 
  rather than 1 (empirically more accurate — 1d is dominated by MC noise and
  discretisation error for FD/binomial).


> Note: The PDE_FD engine (what we consider the production engine) supports all 5 Greeks
> for all barrier configs.

In [ ]:
# ── Greeks for the DO call (European, continuous) ──────────────────────
# Engine-specific availability (with auto-selected greek method):
#   - BSM:        delta, gamma, vega, rho via NUMERICAL bump-and-reprice;
#                 theta on barrier specs uses the analytical engine's
#                 PDE-identity route by default
#   - Binomial:   delta, gamma, theta via TREE; vega unavailable; rho via NUMERICAL
#   - PDE_FD:     delta, gamma, theta via GRID; vega and rho via NUMERICAL
#   - MC:         delta, vega, theta, rho via NUMERICAL bump; gamma blocked

ov_bsm = dp.OptionValuation(underlying, spec_doc, dp.PricingMethod.BSM)
ov_binom = dp.OptionValuation(underlying, spec_doc, dp.PricingMethod.BINOMIAL)
ov_pde = dp.OptionValuation(underlying, spec_doc, dp.PricingMethod.PDE_FD)
ov_mc = dp.OptionValuation(gbm, spec_doc, dp.PricingMethod.MONTE_CARLO)


def _safe(ov, accessor):
    """Call an OV accessor; return NaN if the call raises."""
    try:
        return getattr(ov, accessor)()
    except dp.DerivativesPricingError:
        return float("nan")


def _fmt(x, decimals=4):
    """Format a number, showing 'n/a' for NaN."""
    return "n/a" if math.isnan(x) else f"{x:.{decimals}f}"


print(f"{'Engine':<14} {'PV':>10} {'Δ':>10} {'Γ':>10} {'Vega':>10} {'Θ/day':>10}")
print("-" * 64)
for name, ov in [
    ("BSM", ov_bsm),
    ("Binomial", ov_binom),
    ("PDE_FD", ov_pde),
    ("Monte Carlo", ov_mc),
]:
    pv = _safe(ov, "present_value")
    d = _safe(ov, "delta")
    g = _safe(ov, "gamma")
    v = _safe(ov, "vega")
    th = _safe(ov, "theta")
    print(f"{name:<14} {_fmt(pv):>10} {_fmt(d):>10} {_fmt(g):>10} {_fmt(v):>10} {_fmt(th, 6):>10}")

Engine                 PV          Δ          Γ       Vega      Θ/day
----------------------------------------------------------------
BSM                9.9234     0.6884     0.0067     0.2016  -0.010006
Binomial           9.9262     0.6882     0.0067        n/a  -0.010017
PDE_FD             9.9234     0.6883     0.0067     0.2016  -0.010010
Monte Carlo        9.9658     0.6931        n/a     0.2007  -0.010089


## 6) Discrete Monitoring — European

Under **discrete** monitoring the barrier is checked only at scheduled
observation dates $t_1 < t_2 < \dots < t_M$.  Spot can dip past the
barrier between observations and recover without any consequence — the
contract is alive iff none of the $M$ observations breaches the barrier.
This makes a discretely-monitored KO **more valuable** and a discretely-
monitored KI **less valuable** than their continuously-monitored
equivalents at the same nominal $H$.

The package supports two ways to specify the schedule:

- `num_observations=M` — equally-spaced dates at $t_i = iT/M$ for
  $i = 1, \dots, M$ (the pricing date is **excluded**, matching the
  academic convention used by Broadie-Glasserman-Kou and Boyle-Tian).
- `monitoring_dates=(d1, d2, ...)` — explicit calendar dates.

Each engine handles the discrete schedule differently.

In [ ]:
# ── Discrete European DO call (12 observations) ───────────────
M_OBS = 12
spec_doc_disc = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=K,
    maturity=maturity,
    barrier=H_DOWN,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.DISCRETE,
    num_observations=M_OBS,
)
spec_dic_disc = dc_replace(spec_doc_disc, action=dp.BarrierAction.IN)

### 6.1 BSM analytical — Broadie-Glasserman-Kou continuity correction

Closed-form formulas under discrete monitoring are intractable.  But
Broadie-Glasserman-Kou (1997) show that for $M$ equally-spaced
observations the discrete-monitoring price is **well approximated**, to
leading order, by the **continuous**-monitoring price evaluated at a
**shifted** barrier:

$$H_{\text{adj}} = H \cdot \exp\!\left(\pm\beta\,\sigma\sqrt{T/M}\right)
  \qquad \beta \approx 0.5826$$

The sign is $+$ for UP barriers (push the barrier up to make it harder
to hit) and $-$ for DOWN barriers (push the barrier down).  $\beta = -\zeta(1/2)/\sqrt{2\pi}$
comes from the expected overshoot of a random walk past a barrier.

**Accuracy.**  The shift kills the leading $O(1/\sqrt{M})$
discrete-vs-continuous monitoring error.  BGK 1997 Theorem 1.1
proves the residual is $o(1/\sqrt{M})$ — i.e. shrinks *strictly
faster* than $1/\sqrt{M}$ as $M \to \infty$.  In practice the
accuracy is contract-dependent. Stress configurations — barriers
close to spot, or contracts where the barrier sits in the part of the
payoff profile that dominates the price (e.g. a deep-ITM DO put with
   near-ATM strike) — can show a few percent error.
   
So treat BSM-BGK as a strong first cut for typical configurations.  When
precision matters at low $M$ on stress contracts, prefer PDE_FD.

In the package: when `BarrierSpec.monitoring = DISCRETE` is paired with
`PricingMethod.BSM`, the analytical engine internally applies the BGK
shift to $H$ and then evaluates the standard continuous Reiner-Rubinstein
formulas at $H_{\text{adj}}$.

In [ ]:
bsm_doc_disc = dp.OptionValuation(underlying, spec_doc_disc, dp.PricingMethod.BSM).present_value()
bsm_dic_disc = dp.OptionValuation(underlying, spec_dic_disc, dp.PricingMethod.BSM).present_value()

print(f"── BSM analytical (discrete, M={M_OBS}, DOWN call) ──")
print(f"  Down-and-out:  {bsm_doc_disc:.6f}   (vs continuous {bsm_doc:.6f})")
print(f"  Down-and-in:   {bsm_dic_disc:.6f}   (vs continuous {bsm_dic:.6f})")
print(f"  Sum:           {bsm_doc_disc + bsm_dic_disc:.6f}   (vanilla {bsm_vanilla:.6f})")

── BSM analytical (discrete, M=12, DOWN call) ──
  Down-and-out:  10.567137   (vs continuous 9.923414)
  Down-and-in:   0.556625   (vs continuous 1.200348)
  Sum:           11.123762   (vanilla 11.123762)


### 6.2 Binomial — half-step CRR-layer alignment

Discrete monitoring uses the same CRR backward induction as the
continuous case (§3.3), but the barrier kill / activation rule fires
**only at observation dates**.  Each monitoring date is **snapped to the
nearest tree time-step index**: at those indices, lattice values where
`spots <= H` (DOWN) or `spots >= H` (UP) are zeroed (KO) or absorbed
into the KI lattice (KI).  All other time steps run plain backward
induction without touching the barrier state.


**Half-step alignment.**
`_BinomialBarrierValuation._resolve_effective_num_steps` inflates the
user-supplied `num_steps` to a larger $N$ chosen from the half-step
Boyle-Lau-style sequence

$$N = \left\lfloor \frac{(i + \tfrac{1}{2})^2 \sigma^2 T}{(\ln(H/S))^2} \right\rfloor
  \quad \text{for some integer } i \ge 0.$$

This is a **half-step alignment heuristic**: the
implementation searches over that sequence and picks the first candidate
large enough to use as the effective tree depth.  The goal is to make the
barrier sit *approximately* halfway between adjacent CRR layers, which
materially reduces the leading placement bias from having $H$ sit too close
to one side's layer.

This is the binomial-probability analog of the 
**Cheuk-Vorst (1996) / Boyle-Tian (1998)** half-step grid placement 
used by the FD engine (§6.3): both place $H$ midway between spatial layers to avoid the
leading-order bias from $H$ sitting on or near a node.

The package's discrete-barrier binomial default is `num_steps=3000`,
which keeps enough CRR finite-step headroom for small-price reverse
barriers (e.g. UOC with $H$ close to spot) where the residual
$O(1/N)$ CRR convergence error is most visible.

In [ ]:
binom_doc_disc = dp.OptionValuation(
    underlying, spec_doc_disc, dp.PricingMethod.BINOMIAL
).present_value()
binom_dic_disc = dp.OptionValuation(
    underlying, spec_dic_disc, dp.PricingMethod.BINOMIAL
).present_value()

print(f"── Binomial (discrete, M={M_OBS}, DOWN call) ──")
print(
    f"  Down-and-out:  {binom_doc_disc:.6f}   "
    f"(BSM-discrete: {bsm_doc_disc:.6f},  diff {binom_doc_disc - bsm_doc_disc:+.6f}  "
    f"({(binom_doc_disc - bsm_doc_disc) / bsm_doc_disc:+.2%}))"
)
print(
    f"  Down-and-in:   {binom_dic_disc:.6f}   "
    f"(BSM-discrete: {bsm_dic_disc:.6f},  diff {binom_dic_disc - bsm_dic_disc:+.6f}  "
    f"({(binom_dic_disc - bsm_dic_disc) / bsm_dic_disc:+.2%}))"
)

── Binomial (discrete, M=12, DOWN call) ──
  Down-and-out:  10.553778   (BSM-discrete: 10.567137,  diff -0.013359  (-0.13%))
  Down-and-in:   0.569217   (BSM-discrete: 0.556625,  diff +0.012592  (+2.26%))


### 6.3 PDE_FD — half-step grid alignment

Continuous-monitoring KO truncates the grid at the barrier — the barrier
becomes a domain boundary.  This approach **does not transfer** to discrete
monitoring: we need the alive zone to extend across the barrier between
observation dates (since spot is allowed to dip past $H$ and recover),
and we only kill paths *at* observation dates.

Instead, for discrete monitoring the package uses the **full** spot
grid (no truncation) and applies the barrier reset as an instantaneous
jump at each observation date — analogous to how a discrete dividend
is handled.

Following the **Cheuk-Vorst (1996) / Boyle-Tian (1998)**-style midpoint-placement idea for discretely
monitored barriers, the `derivatives-pricing` package's discrete-barrier
PDE solver picks $S_{\min}, S_{\max}$ and the grid spacing such that
$H$ lands exactly between two log-spot nodes (`anchor_half_step=True`).
Combined with **explicit Hull** time-stepping at high temporal
resolution, this gives clean, well-behaved discrete-barrier prices and
greeks: each step is a single vectorised matrix-vector multiply (no
inner linear solve), so we can crank `time_steps` up cheaply, and
Hull's adaptive $dz = \sigma\sqrt{3\,\Delta\tau}$ keeps the spatial
step in CFL automatically.

`PDEParams.for_barriers(monitoring=BarrierMonitoring.DISCRETE)` ships
with `1200 × 3000` (spot × time), `EXPLICIT_HULL` time stepping,
`INTRINSIC` early-exercise projection for Americans, and half-step
anchoring on by default.

Note: What the package calls `EXPLICIT_HULL` is finance-style explicit FD 
with implicit discounting. As opposed to a pure
`EXPLICIT` scheme (fully forward Euler), which the package also supports.

In [ ]:
pde_doc_disc = dp.OptionValuation(
    underlying, spec_doc_disc, dp.PricingMethod.PDE_FD
).present_value()
pde_dic_disc = dp.OptionValuation(
    underlying, spec_dic_disc, dp.PricingMethod.PDE_FD
).present_value()

print(f"── PDE_FD (discrete, M={M_OBS}, DOWN call) ──")
print(
    f"  Down-and-out:  {pde_doc_disc:.6f}   "
    f"(BSM-discrete: {bsm_doc_disc:.6f},  diff {pde_doc_disc - bsm_doc_disc:+.6f}  "
    f"({(pde_doc_disc - bsm_doc_disc) / bsm_doc_disc:+.2%}))"
)
print(
    f"  Down-and-in:   {pde_dic_disc:.6f}   "
    f"(BSM-discrete: {bsm_dic_disc:.6f},  diff {pde_dic_disc - bsm_dic_disc:+.6f}  "
    f"({(pde_dic_disc - bsm_dic_disc) / bsm_dic_disc:+.2%}))"
)

── PDE_FD (discrete, M=12, DOWN call) ──
  Down-and-out:  10.563922   (BSM-discrete: 10.567137,  diff -0.003215  (-0.03%))
  Down-and-in:   0.559054   (BSM-discrete: 0.556625,  diff +0.002429  (+0.44%))


### 6.4 Monte Carlo — monitoring-date injection

For discrete monitoring there is no continuity-correction issue (no
intra-step hits to worry about) provided the monitoring dates **are
themselves grid nodes**.  The package builds the standard MC time grid
from `SimulationConfig` and then **injects the monitoring dates** as
extra nodes if they aren't already present.  Each path tests the barrier
only at those specific nodes.

Concretely:

1. Build the time grid $\{0, t_1, t_2, \dots, T\}$ from `SimulationConfig`.
2. Insert each monitoring date $\tau_j$ that isn't already on the grid.
3. Simulate GBM paths on the augmented grid as usual.
4. For each path, check the barrier **only at monitoring nodes**.  Record
   the first hit time (if any).  Apply rebate / payoff per the
   knock-in / knock-out rules.

No Brownian-bridge correction is applied (since intra-monitoring step crossings 
don't matter).

In [ ]:
mc_doc_disc = dp.OptionValuation(gbm, spec_doc_disc, dp.PricingMethod.MONTE_CARLO).present_value()
mc_dic_disc = dp.OptionValuation(gbm, spec_dic_disc, dp.PricingMethod.MONTE_CARLO).present_value()

print(f"── Monte Carlo (discrete, M={M_OBS}, DOWN call) ──")
print(
    f"  Down-and-out:  {mc_doc_disc:.6f}   "
    f"(BSM-discrete: {bsm_doc_disc:.6f},  diff {mc_doc_disc - bsm_doc_disc:+.6f}  "
    f"({(mc_doc_disc - bsm_doc_disc) / bsm_doc_disc:+.2%}))"
)
print(
    f"  Down-and-in:   {mc_dic_disc:.6f}   "
    f"(BSM-discrete: {bsm_dic_disc:.6f},  diff {mc_dic_disc - bsm_dic_disc:+.6f}  "
    f"({(mc_dic_disc - bsm_dic_disc) / bsm_dic_disc:+.2%}))"
)

── Monte Carlo (discrete, M=12, DOWN call) ──
  Down-and-out:  10.599811   (BSM-discrete: 10.567137,  diff +0.032674  (+0.31%))
  Down-and-in:   0.562913   (BSM-discrete: 0.556625,  diff +0.006288  (+1.13%))


### 6.5 Discrete European cross-engine table

In [ ]:
print(f"{'Engine':<14} {'DO call':>12} {'DI call':>12} {'Sum':>12}")
print("-" * 52)
print(
    f"{'BSM (BGK)':<14} {bsm_doc_disc:>12.6f} {bsm_dic_disc:>12.6f} {bsm_doc_disc + bsm_dic_disc:>12.6f}"
)
print(
    f"{'Binomial':<14} {binom_doc_disc:>12.6f} {binom_dic_disc:>12.6f} {binom_doc_disc + binom_dic_disc:>12.6f}"
)
print(
    f"{'PDE_FD':<14} {pde_doc_disc:>12.6f} {pde_dic_disc:>12.6f} {pde_doc_disc + pde_dic_disc:>12.6f}"
)
print(
    f"{'Monte Carlo':<14} {mc_doc_disc:>12.6f} {mc_dic_disc:>12.6f} {mc_doc_disc + mc_dic_disc:>12.6f}"
)
print(f"\nVanilla call (BSM):        {bsm_vanilla:.6f}  (parity check)")
print(f"Continuous DO + DI (BSM):  {bsm_doc + bsm_dic:.6f}")
print(
    f"Discrete DO call > Continuous DO call?  {bsm_doc_disc > bsm_doc}  "
    f"(yes — fewer hit opportunities)"
)

Engine              DO call      DI call          Sum
----------------------------------------------------
BSM (BGK)         10.567137     0.556625    11.123762
Binomial          10.553778     0.569217    11.122995
PDE_FD            10.563922     0.559054    11.122976
Monte Carlo       10.599811     0.562913    11.162724

Vanilla call (BSM):        11.123762  (parity check)
Continuous DO + DI (BSM):  11.123762
Discrete DO call > Continuous DO call?  True  (yes — fewer hit opportunities)


## 7) Discrete American

Same three engines as continuous American (BSM analytical doesn't apply).
The **only difference** vs §4 is that the barrier check happens only at
monitoring dates — every engine handles this internally:

- **Binomial**: monitoring dates snap to tree layers (as in §6.2);
  between monitoring layers the tree just does early-exercise comparisons
  with no barrier interaction.
- **PDE_FD**: full grid with half-step barrier anchoring; the
  free-boundary $V \ge \text{intrinsic}$ check runs at every time step,
  and the barrier reset is applied only at monitoring dates.  KI uses
  the same two-surface coupled solver as continuous American KI.
- **Monte Carlo**: barrier is checked at injected monitoring nodes only;
  LSM regression and exercise comparison run at every grid node.

As in §4 we use the **DI put** as the running example to keep all three
engines on a contract that converges cleanly.

In [ ]:
spec_dip_disc_eu = dc_replace(
    spec_dic_disc,
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
)
spec_dip_disc_am = dc_replace(spec_dip_disc_eu, exercise_type=dp.ExerciseType.AMERICAN)

binom_dip_disc_eu = dp.OptionValuation(
    underlying, spec_dip_disc_eu, dp.PricingMethod.BINOMIAL
).present_value()
binom_dip_disc_am = dp.OptionValuation(
    underlying, spec_dip_disc_am, dp.PricingMethod.BINOMIAL
).present_value()
pde_dip_disc_eu = dp.OptionValuation(
    underlying, spec_dip_disc_eu, dp.PricingMethod.PDE_FD
).present_value()
pde_dip_disc_am = dp.OptionValuation(
    underlying, spec_dip_disc_am, dp.PricingMethod.PDE_FD
).present_value()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mc_dip_disc_eu = dp.OptionValuation(
        gbm, spec_dip_disc_eu, dp.PricingMethod.MONTE_CARLO
    ).present_value()
    mc_dip_disc_am = dp.OptionValuation(
        gbm, spec_dip_disc_am, dp.PricingMethod.MONTE_CARLO
    ).present_value()

bsm_dip_disc_eu = dp.OptionValuation(
    underlying, spec_dip_disc_eu, dp.PricingMethod.BSM
).present_value()

print(f"── Discrete DI put (M={M_OBS}, monthly observations) ──")
print(f"{'Engine':<16} {'European':>12} {'American':>12} {'EE prem':>12}")
print("-" * 54)
print(f"{'BSM (BGK)':<16} {bsm_dip_disc_eu:>12.6f} {'n/a':>12} {'n/a':>12}")
print(
    f"{'Binomial':<16} {binom_dip_disc_eu:>12.6f} {binom_dip_disc_am:>12.6f} "
    f"{binom_dip_disc_am - binom_dip_disc_eu:>+12.6f}"
)
print(
    f"{'PDE_FD':<16} {pde_dip_disc_eu:>12.6f} {pde_dip_disc_am:>12.6f} "
    f"{pde_dip_disc_am - pde_dip_disc_eu:>+12.6f}"
)
print(
    f"{'Monte Carlo':<16} {mc_dip_disc_eu:>12.6f} {mc_dip_disc_am:>12.6f} "
    f"{mc_dip_disc_am - mc_dip_disc_eu:>+12.6f}"
)

── Discrete DI put (M=12, monthly observations) ──
Engine               European     American      EE prem
------------------------------------------------------
BSM (BGK)            7.332575          n/a          n/a
Binomial             7.407508     7.727039    +0.319531
PDE_FD               7.409377     7.728734    +0.319357
Monte Carlo          7.418882     7.716598    +0.297717


## 8) Summary

| Engine | Continuous EU | Continuous AM | Discrete EU | Discrete AM |
|---|---|---|---|---|
| **BSM analytical** | Reiner-Rubinstein closed form | ✗ (no closed form) | BGK continuity correction | ✗ |
| **Binomial** | Boyle-Lau on-node alignment | Boyle-Lau on-node + free-boundary | Boyle-Tian half-step analog | Half-step + free-boundary |
| **PDE_FD** | Truncated grid (KO) + parity (KI) | Truncated KO + 2-surface KI | Full grid + Boyle-Tian half-step | Full grid + half-step + free-boundary + 2-surface KI |
| **Monte Carlo** | Brownian-bridge correction | LSM with barrier-aware basis | Monitoring-date injection | LSM + monitoring-date injection |

**When to use which engine:**
- **TL;DR: PDE_FD is the production engine.** It supports continuous 
  and discrete monitoring, European and American exercise, discrete 
  dividends, and non-flat rate/dividend curves, and it stays accurate 
  in the hard cases — spot near the barrier, barrier truncating the 
  payoff profile, etc. With default parameters it produces highly 
  accurate PVs and greeks, validated against academic benchmarks 
  (see tests).

Further comments:
- **BSM analytical** — by far the fastest; first-line tool for European
  barriers under GBM with simple curves and no path-dependent extras.
- **Binomial** — good for American single-barrier problems where
  Boyle-Lau alignment behaves well (barrier not pathologically close to
  spot).  Engine-native greeks via TREE.
- **PDE_FD** — the workhorse for "everything else": discrete monitoring,
  discrete dividends, near-spot barriers where Binomial can't align.
  Engine-native greeks via GRID.
- **Monte Carlo** — best when path dependencies stack up or the underlying 
  isn't GBM (supports jump diffusion for example, though only discrete 
  monitoring currently). 

**What's not yet covered (see notebook 06b):**

- **Truncated-payoff KO Americans** (DO put, UO call) — the harder
  numerical regime we deferred from §4 / §7.
- Behaviour when the barrier is **extremely close to spot** (Boyle-Lau
  step-cap warnings, PDE half-step robustness, etc.)
- **Inception-triggered** barriers (KO/KI specs where the barrier is
  already on the wrong side of spot at $t=0$): how each engine collapses
  to the appropriate deterministic instrument and how the OV-level
  short-circuit handles greeks under that regime.
- Numerical greek subtleties: spot-bump caps near barrier, the 7-day
  theta default for barriers, MC barrier gamma noise.

**References**

- Reiner, E. and Rubinstein, M. (1991). "Breaking Down the Barriers",
  *Risk* **4**(8), 28–35.
- Hull, J. C. *Options, Futures, and Other Derivatives*, §26.9.
- Boyle, P. P. and Lau, S. H. (1994). "Bumping Up Against the Barrier
  With the Binomial Method", *Journal of Derivatives* **1**(4), 6–14.
- Boyle, P. P. and Tian, Y. (1998). "An Explicit Finite Difference
  Approach to the Pricing of Barrier Options", *Applied Mathematical
  Finance* **5**, 17–43.
- Broadie, M., Glasserman, P. and Kou, S. (1997). "A Continuity
  Correction for Discrete Barrier Options", *Mathematical Finance*
  **7**(4), 325–349.
- Beaglehole, D. R., Dybvig, P. H. and Zhou, G. (1997). "Going to
  Extremes: Correcting Simulation Bias in Exotic Option Valuation",
  *Financial Analysts Journal* **53**(1), 62–68.
- Longstaff, F. A. and Schwartz, E. S. (2001). "Valuing American Options
  by Simulation: A Simple Least-Squares Approach", *Review of Financial
  Studies* **14**(1), 113–147.